<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/clean_transcripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install pyspellchecker


In [183]:
!pip install wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 12.5 MB/s eta 0:00:00


In [184]:
import pandas as pd
import numpy as np
import sys
import re
import os
import re

import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
import pickle
import json

from pathlib import Path
import spacy
from spellchecker import SpellChecker

from multiprocessing import Pool
from multiprocessing import cpu_count
import math
import time
import random
from wordfreq import zipf_frequency



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [32]:
from nltk.corpus import words
nltk.download('words')


[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [13]:
  >>> import nltk
  >>> nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Basic Cleaning Steps

In [15]:
def lower_tokenize_regex(transcript_path, clean_transcript_pathname):

  with open(transcript_path, 'r', encoding='utf8', errors='replace') as infile:
    text_data = infile.read()
    text_data = text_data.lower()
    cleaned_text = re.sub(r"[^a-z ]", '', text_data)
    cleaned_text = word_tokenize(cleaned_text)

  with open(clean_transcript_pathname, "w", encoding="utf-8") as outfile:
    json.dump(cleaned_text, outfile)

In [16]:
#apply basic cleaning to each transcript
def full_basic_clean(transcripts_path, output_folder):
  for filename in  os.listdir(transcripts_path):
    full_path= transcripts_path + '/' +filename

    name , _ , txt = filename.partition('.')

    new_file_path = output_folder + '/' +name +'.json'
    lower_tokenize_regex(full_path, new_file_path)


In [ ]:
#sanity check on a single example
# example_path = '/content/drive/MyDrive/Capital_trials/women/example/Angelina Rodriguez.txt'
# example_output = '/content/drive/MyDrive/Capital_trials/women/example/Angelina Rodriguez_bc.json'
# lower_tokenize_regex(example_path, example_output)

In [ ]:
# full_basic_clean('/content/drive/MyDrive/Capital_trials/women', '/content/drive/MyDrive/Capital_trials/basic_clean/women')
# full_basic_clean('/content/drive/MyDrive/Capital_trials/men', '/content/drive/MyDrive/Capital_trials/basic_clean/men')
# full_basic_clean('/content/drive/MyDrive/Capital_trials/joint', '/content/drive/MyDrive/Capital_trials/basic_clean/joint')

Create Dictionary of all words in corpus

In [17]:
#create dict of all words across all transcripts
def create_dict(tokenized_path):
  full_corpus_dict = set()

  for clean_transcript in tokenized_path.glob("*.json"):

      with open(clean_transcript, "r", encoding="utf-8") as f:
          all_transcript_words = json.load(f)
          full_corpus_dict.update(all_transcript_words)

  return full_corpus_dict

In [18]:
w_dict = create_dict(Path('/content/drive/MyDrive/Capital_trials/basic_clean/women'))
m_dict = create_dict(Path('/content/drive/MyDrive/Capital_trials/basic_clean/men'))

In [19]:
w_m_dict = w_dict | m_dict

In [133]:
#check legal terms with lexpredict legal dictionary https://github.com/LexPredict/lexpredict-legal-dictionary/blob/master/en/legal/common_US_terms_top1000.csv
legal_terms_df = pd.read_csv('/content/common_US_terms_top1000.csv')
legal_list = legal_terms_df.Term.to_list()


Find outlier words in the corpus using multiple relevant vocab lists

In [144]:
nltk_vocab = set(w.lower() for w in words.words())
legal_vocab = (set(legal_list) - nltk_vocab)

In [149]:
outlier_w =  w_dict - nltk_vocab

outlier_m = m_dict - nltk_vocab

outlier_w =  outlier_w - legal_vocab

outlier_m = outlier_m - legal_vocab

In [182]:

print(f"Number womens unqiue words {len(w_dict)} Number of womens outliers {len(outlier_w)}")

print(f"Number mens unqiue words {len(m_dict)} Number of mens outliers {len(outlier_m)}")

print(f"Total unqiue words {len(w_dict | m_dict )} total outliers {len(outlier_m | outlier_w)}")




Number womens unqiue words 91764 Number of womens outliers 67621
Number mens unqiue words 324949 Number of mens outliers 299530
Total unqiue words 370303 total outliers 340369


Remove outliers that are caused by missing space

In [166]:
def likely_missing_space(token: str, vocab: nltk_vocab, min_part_len: int = 2) -> bool:
    t = re.sub(r"[^a-z']", "", token.lower())
    if len(t) < 2 * min_part_len or t in vocab:
        return False
    if t in vocab:  # already a known word
        return False

    for i in range(min_part_len, len(t) - min_part_len + 1):
        left, right = t[:i], t[i:]
        if left in vocab and right in vocab:
            return True
    return False

In [167]:
def seperate_missing_space_outliers(outliers, likely_missing_space_set, not_missing_space_set, vocab: nltk_vocab):
  for w in outliers:
      if likely_missing_space(w, vocab):
          likely_missing_space_set.add(w)
      else:
          not_missing_space_set.add(w)

  print("likely missing space:", len(likely_missing_space_set))
  print("not missing space:", len(not_missing_space_set))

In [168]:
likely_missing_space_w = set()
remaining_outliers_w = set()

likely_missing_space_m = set()
remaining_outliers_m = set()


In [169]:
seperate_missing_space_outliers(outlier_w, likely_missing_space_w, remaining_outliers_w, nltk_vocab )


likely missing space: 25520
not missing space: 42101


In [170]:
seperate_missing_space_outliers(outlier_m, likely_missing_space_m, remaining_outliers_m, nltk_vocab )


likely missing space: 74971
not missing space: 224559


In [180]:
remaining_outliers_w_m = remaining_outliers_w| remaining_outliers_m
len(remaining_outliers_w_m)

251741

In [181]:

print(f"Number womens unqiue words {len(w_dict)} Number of womens outliers {len(remaining_outliers_w)}")

print(f"Number mens unqiue words {len(m_dict)} Number of mens outliers{len(remaining_outliers_m)}")

print(f"Total unqiue words {len(w_dict | m_dict )} total outliers {len(remaining_outliers_w_m)}")

Number womens unqiue words 91764 Number of womens outliers 42101
Number mens unqiue words 324949 Number of mens outliers224559
Total unqiue words 370303 total outliers 251741


In [188]:
likely_missing_space_w_m = likely_missing_space_m | likely_missing_space_w
len(likely_missing_space_w_m)

88628

For ourliers caused by missing space, create map of splits

In [99]:
def best_two_word_split(token: str, vocab: set[str], word_freq: dict[str, float] | None = None, min_part_len: int = 2):
    t = re.sub(r"[^a-z']", "", token.lower())
    n = len(t)
    if n < 2 * min_part_len or t in vocab:
        return None

    best = None
    best_score = float("-inf")

    for i in range(min_part_len, n - min_part_len + 1):
        left, right = t[:i], t[i:]
        if left in vocab and right in vocab:
            if word_freq:
                score = word_freq.get(left, 1e-12) * word_freq.get(right, 1e-12)
            else:
                score = -abs(len(left) - len(right))  # fallback heuristic
            if score > best_score:
                best_score = score
                best = (left, right)

    return best


In [177]:
sample_missing_space = random.sample(list(likely_missing_space_w_m), 2000)

In [185]:
start = time.time()

split_map = {}
for w in likely_missing_space_w_m:
    split = best_two_word_split(w, nltk_vocab, word_freq=None)
    if split is not None:
        split_map[w] = split
split_text_map = {k: f"{v[0]} {v[1]}" for k, v in split_map.items()}

print(f"splits for {len(split_map)} tokens")

end = time.time()
print(f"took  {end- start} sec")

splits for 88628 tokens
took  1.5462627410888672 sec


In [187]:
len(split_map.keys())

88628

Spellcheck outliers not split by space

In [203]:
def spellcheck(remaining_outliers):
  spellchecked_likely_outlier = set()
  spellchecked_likely_word = set()

  for word in remaining_outliers:
    freq = zipf_frequency(word, 'en')
    if freq < 2.5:
      spellchecked_likely_outlier.add(word)
    else:
      spellchecked_likely_word.add(word)


  return spellchecked_likely_outlier, spellchecked_likely_word

In [204]:
spellchecked_likely_outlier_w , spellchecked_likely_word_w = spellcheck(remaining_outliers_w)

spellchecked_likely_outlier_m , spellchecked_likely_word_m = spellcheck(remaining_outliers_m)


In [205]:
print(f"Number womens unqiue words {len(w_dict)} Number of womens outliers remaining {len(spellchecked_likely_outlier_w)}")

print(f"Number mens unqiue words {len(m_dict)} Number of mens outliers remaining {len(spellchecked_likely_outlier_m)}")

print(f"Total unqiue words {len(w_dict | m_dict )} total outliers remaining {len(spellchecked_likely_outlier_m | spellchecked_likely_outlier_w)}")

Number womens unqiue words 91764 Number of womens outliers remaining 34162
Number mens unqiue words 324949 Number of mens outliers remaining 215966
Total unqiue words 370303 total outliers remaining 241940


In [206]:
spellchecked_likely_outlier_m

{'fewminu',
 'totokindoff',
 'ojtside',
 'themrightso',
 'irvingqis',
 'sittingininn',
 'leastleasnow',
 'frombegins',
 'thepmrecord',
 'cappels',
 'generallyeneralldiagnosedagaastheresultoff',
 'reidmce',
 'aphso',
 'iln',
 'tomm',
 'thethedefendantthebea',
 'processesh',
 'lnlo',
 'patternpmanalysis',
 'angleqin',
 'meanseemedlikejuste',
 'againpmthats',
 'oflingerprints',
 'moond',
 'aanyim',
 'kindnn',
 'youamimagine',
 'hedaughterhe',
 'judgeduandn',
 'callheomeantlinn',
 'mosjy',
 'qeis',
 'thatthaapartmentmentrightight',
 'duringi',
 'fulltimeparttime',
 'lutts',
 'receivedpmseveral',
 'seemsthat',
 'thatultimatelytd',
 'safekeepers',
 'hasdrained',
 'haventyouayes',
 'courtspmproposed',
 'tiitdoeso',
 'neighborgrwho',
 'rdtype',
 'anotheramparty',
 'aad',
 'testq',
 'athreeperson',
 'numberayeah',
 'abitilynot',
 'fiveeighths',
 'evidwce',
 'aboutamtwo',
 'sinceqso',
 'studentayesqdid',
 'minnies',
 'iqentification',
 'appropriater',
 'showsaqyes',
 'renigged',
 'behaviorsthat'

Create Dictionary of misspelled words and possible replacements, save words that do not have good replacement options seperately


In [51]:
def create_replacement_dict(outlier_set):
  replacement_dict = {}

  for word in (outlier_set):

    most_likely = spell.correction(word)
    # other_options = spell.candidates(word)

    replacement_dict[word] = most_likely

  return replacement_dict

In [53]:
#create a smaller subset of outliers to test
sample_n = 2000

sample = random.sample(list(outlier_w), min(sample_n, len(list(outlier_w))))


In [60]:
spell = SpellChecker()

start = time.time()

replacement_dict = create_replacement_dict(sample)

total_time = time.time() - start
print(f"Code block took: {total_time:.6f} seconds")


KeyboardInterrupt: 

In [ ]:
replacement_dict.keys()

NameError: name 'replacement_dict' is not defined

In [ ]:
candiadates_dict

{'brassish': {'bassist',
  'brackish',
  'brandish',
  "brass's",
  'brasses',
  'brassica',
  'brassie',
  'brassier',
  'brassies',
  'brassiest',
  'brassily'},
 'baldachino': {'baldachino'},
 'theodosia': {'theodora', 'theodoric', 'theodosius'},
 'growsome': {'gruesome'},
 'batikuling': None,
 'combustibly': {'combustibly'},
 'hospitant': {'hesitant', 'hospital', 'hospitalet', 'hospitals', 'oscitant'},
 'pseudohydrophobia': None,
 'pterodactyl': {'pterodactyl'},
 'talonavicular': None,
 'forgedly': {'forcedly'},
 'baccheion': None,
 'angioneurotic': None,
 'oversapless': None,
 'adherently': {'adherent',
  "adherent's",
  'adherents',
  'coherently',
  'inherently'},
 'agunah': {'agana', 'aguish', 'gunyah'},
 'riksmaal': None,
 'unmaturing': {'maturing'},
 'erysibe': {'erosive'},
 'goodyish': {'goodish'},
 'kolinski': {'kolinsky'},
 'tartareous': {'tartarous'},
 'carbanilic': None,
 'ferociousness': {'ferociousness'},
 'elsewhither': None,
 'unappointableness': None,
 'diastematic'

In [ ]:
with open('/content/drive/MyDrive/Capital_trials/basic_clean/spell/replace_most_likely_full.json', "w", encoding="utf-8") as outfile:
  json.dump(replacement_dict, outfile)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Capital_trials/basic_clean/spell/replace_most_likely_full.json'

In [ ]:
  with open('/content/drive/MyDrive/Capital_trials/basic_clean/spell/replace_candidates.json', "w", encoding="utf-8") as outfile:
    json.dump(candiadates_dict, outfile)

TypeError: Object of type set is not JSON serializable